# Black-Box Optimisation Capstone: Final Analysis

This notebook calculates the final results directly from the committed query history and derived initial incumbents. Its outputs are saved so the result is visible on GitHub without executing the notebook.

In [1]:
from pathlib import Path
import csv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
with (ROOT / 'data' / 'query_history.csv').open() as stream:
    history = list(csv.DictReader(stream))
with (ROOT / 'data' / 'initial_best.csv').open() as stream:
    initial = {int(row['function']): float(row['best_initial']) for row in csv.DictReader(stream)}
print(f'Loaded {len(history)} weekly evaluations and {len(initial)} initial incumbents.')


Loaded 104 weekly evaluations and 8 initial incumbents.


In [2]:
expected_dimensions = {1: 2, 2: 2, 3: 3, 4: 4, 5: 4, 6: 5, 7: 6, 8: 8}
seen = set()
for row in history:
    key = (int(row['round']), int(row['function']))
    assert key not in seen, f'Duplicate round/function record: {key}'
    seen.add(key)
    function = int(row['function'])
    coordinates = [row[f'x{i}'] for i in range(1, 9) if row[f'x{i}']]
    assert len(coordinates) == expected_dimensions[function]
    assert all(value.startswith('0.') and len(value.split('.')[-1]) == 6 for value in coordinates)
    assert all(0 <= float(value) <= 0.999999 for value in coordinates)
assert len(seen) == 13 * 8
print('Validated 104 unique queries: dimensions, bounds and six-decimal formatting are correct.')


Validated 104 unique queries: dimensions, bounds and six-decimal formatting are correct.


In [3]:
results = []
for function in range(1, 9):
    rows = [row for row in history if int(row['function']) == function]
    best_weekly = max(rows, key=lambda row: float(row['output']))
    weekly_value = float(best_weekly['output'])
    if initial[function] >= weekly_value:
        best_value, source = initial[function], 'initial data'
    else:
        best_value, source = weekly_value, f"Week {best_weekly['round']}"
    results.append((function, initial[function], best_value, source))

print('function | best initial | final best | first achieved')
for function, initial_value, final_value, source in results:
    print(f'{function:>8} | {initial_value:>12.8g} | {final_value:>12.8g} | {source}')
assert all(final >= initial_value for _, initial_value, final, _ in results)


function | best initial | final best | first achieved
       1 | 7.7108751e-16 | 0.00013624337 | Week 13
       2 |   0.61120522 |   0.68594476 | Week 9
       3 | -0.034835313 | -0.0028315397 | Week 6
       4 |   -4.0255423 |   0.67892545 | Week 13
       5 |    1088.8596 |    1636.5047 | Week 13
       6 |  -0.71426495 |  -0.30565014 | Week 3
       7 |    1.3649683 |    1.8599092 | Week 13
       8 |     9.598482 |    9.9429688 | Week 12


## Method summary

Early rounds used per-function Gaussian Processes with UCB and Expected Improvement. Later rounds added regression, SVM, neural-network, clustering, PCA and RL perspectives, then shifted toward function-specific local refinement. See the model card, datasheet and living documentation for assumptions and limitations.

## Interpretation

Every final incumbent exceeds the corresponding best initial observation. Because the functions and true optima remain unknown, this demonstrates empirical improvement rather than global optimality. Cross-function output magnitudes are not comparable.